# Vorbereitung
* Packages importieren
* Alte Datein löschen
* Seed für Reproduzierbarkeit definieren

In [0]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import random
import requests
import random
import time
from typing import List, Dict, Tuple

try:
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/churn_labels.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/connection_quality_logs.csv')
    os.remove('/Workspace/Users/nina.merkt@abat.de/test/data/customer_profile.csv')
    os.remove ('/Workspace/Users/nina.merkt@abat.de/test/data/agent_profile.csv')
except FileNotFoundError:
    pass

np.random.seed(42)
random.seed(42)

# Echte Adressen besorgen
API configs

In [0]:
API_CONFIGS = {
    'Schleswig-Holstein': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Adressen_Schleswig_Holstein/FeatureServer/0/query',
        'fields': 'GMD,STR,HNR,POSTPLZ',
        'max_object_id': 948763,
        'field_mapping': {
            'city': 'GMD',
            'street': 'STR',
            'house_number': 'HNR',
            'postal_code': 'POSTPLZ'
        }
    },
    'Hamburg': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Adressen_Hamburg/FeatureServer/0/query',
        'fields': 'portsname,strname,hausnr,plz',
        'max_object_id': 284271,
        'field_mapping': {
            'city': 'portsname',
            'street': 'strname',
            'house_number': 'hausnr',
            'postal_code': 'plz'
        }
    },
    'Sachsen': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Adressen_Sachsen/FeatureServer/0/query',
        'fields': 'gmd,str,hnr,postplz',
        'max_object_id': 986876,
        'field_mapping': {
            'city': 'gmd',
            'street': 'str',
            'house_number': 'hnr',
            'postal_code': 'postplz'
        }
    },
    'Berlin': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Berlin_Adressen/FeatureServer/0/query',
        'fields': 'GmlID,str_name,hnr,plz',
        'max_object_id': 401673,
        'field_mapping': {
            'city': 'GmlID',
            'street': 'str_name',
            'house_number': 'hnr',
            'postal_code': 'plz'
        }
    },
    'Thüringen': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Adressen_TH/FeatureServer/0/query',
        'fields': 'gmd,str,hnr,postplz',
        'max_object_id': 623802,
        'field_mapping': {
            'city': 'gmd',
            'street': 'str',
            'house_number': 'hnr',
            'postal_code': 'postplz'
        }
    },
    'Brandenburg': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/Adressen_wm_Brandenburg/FeatureServer/0/query',
        'fields': 'gmd,str,hnr,postplz',
        'max_object_id': 862811,
        'field_mapping': {
            'city': 'gmd',
            'street': 'str',
            'house_number': 'hnr',
            'postal_code': 'postplz'
        }
    },
    'Nordrhein-Westfalen': {
        'url': 'https://services2.arcgis.com/jUpNdisbWqRpMo35/arcgis/rest/services/nrw_adress/FeatureServer/0/query',
        'fields': 'gmd,str,hnr,plz',
        'max_object_id': 4444063,
        'field_mapping': {
            'city': 'gmd',
            'street': 'str',
            'house_number': 'hnr',
            'postal_code': 'plz'
        }
    }
}

## Holt zufällige Adressen von der API eines Bundeslandes
    
Args:

    state: Bundesland-Name

    count: Anzahl gewünschter Adressen

In [0]:
def fetch_addresses_from_api(state: str, count: int) -> List[str]:
    config = API_CONFIGS[state]
    field_mapping = config['field_mapping']
    addresses = []
    
    # Generiere IDs
    max_object_id = config['max_object_id']
    object_ids = random.sample(range(1, max_object_id + 1), min(count, max_object_id))
    
    # Abrufen in Batches
    batch_size = 100
    for i in range(0, len(object_ids), batch_size):
        batch_ids = object_ids[i:i+batch_size]
        
        params = {
            'where': '1=1',
            'objectIds': ','.join(map(str, batch_ids)),
            'outFields': config['fields'],
            'returnGeometry': 'false',
            'f': 'json'
        }
        
        try:
            response = requests.get(config['url'], params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if 'features' in data:
                for feature in data['features']:
                    attrs = feature.get('attributes', {})
                    
                    # Verwende das field_mapping für korrekten Zugriff
                    street = attrs.get(field_mapping['street'], 'Unbekannte Straße')
                    house_nr = attrs.get(field_mapping['house_number'], '')
                    postal = attrs.get(field_mapping['postal_code'], '')
                    city = attrs.get(field_mapping['city'], '')
                    
                    # Formatiere Adresse
                    address = f"{street} {house_nr}, {postal} {city}"
                    addresses.append(address.strip())
            else:
                # Debug-Info bei fehlenden Features
                print(f"Keine Features in Response für {state}")
                if 'error' in data:
                    print(f"API-Fehler: {data['error']}")
            
            # Rate limiting
            time.sleep(0.2)
            
        except requests.exceptions.RequestException as e:
            print(f"Request-Fehler bei {state}: {e}")
            continue
        except Exception as e:
            print(f"Unerwarteter Fehler bei {state}: {e}")
            continue
    
    return addresses

## Hauptfunktion zum Abrufen von zufälligen deutschen Adressen
    
Args:
    
    n_customers: Anzahl benötigter Adressen

In [0]:
def get_random_german_addresses(n_customers: int) -> List[str]:
    states = list(API_CONFIGS.keys())
    addresses = []
    
    # Verteile Kunden proportional zur Anzahl verfügbarer Adressen
    total_addresses = sum(config['max_object_id'] for config in API_CONFIGS.values())
        
    for state in states:
        proportion = API_CONFIGS[state]['max_object_id'] / total_addresses
        count = int(n_customers * proportion)
            
        if count > 0:
            print(f"Hole {count} Adressen aus {state} (von max. {API_CONFIGS[state]['max_object_id']:,})...")
            state_addresses = fetch_addresses_from_api(state, count)
            addresses.extend(state_addresses)
            print(f" {len(state_addresses)} Adressen erhalten")

    
    # Falls nicht genug Adressen gesammelt wurden, fülle mit Duplikaten
    while len(addresses) < n_customers:
        addresses.append(random.choice(addresses) if addresses else "Placeholder Address")
    
    # Mische die Adressen
    random.shuffle(addresses)
    
    return addresses[:n_customers]

### Qualitätsüberprüfung Adressen
Analysiert die Qualität der Adressen pro Bundesland

In [0]:
def analyze_address_quality():
    for state in API_CONFIGS.keys():
        print(f"\n{'='*60}")
        print(f"Analysiere {state}")
        print(f"{'='*60}")
        
        addresses = fetch_addresses_from_api(state, 50)
        
        valid = 0
        invalid = 0
        
        for addr in addresses:
            if "None" in addr or addr.strip() == ",":
                invalid += 1
                print(f"{addr}")
            else:
                valid += 1
        
        print(f"\nGültig: {valid}/{len(addresses)}")
        print(f"Ungültig: {invalid}/{len(addresses)}")

analyze_address_quality()

# Customer Profile generieren
Mit:
* ID (Format: CUST_#####)
* Namen (zufällige Mischung aus 20 Vor- und Nachnamen)
* Signup Date (rückführend ab dem 1.3.26)
* Plan Tier (welchen Vertrag sie abgeschlossen haben)
* Adresse (vorher aus den APIs geholt, damit reale Adressen verwendet werden)
* Contract Type (Um welche Zahlung es sich handelt: monatliche, jährliche oder alle 2 Jahre)
* Payment Method (Kredit Karte, Bank Überweisung, Scheck per Post, elektronischer Scheck)
* Autopay (zahlt der Kunde automatisch)
* Monthly Bill (Menge, wie viel monatlich bezahlt werden muss)
* Data Usage letzten Monat

In [0]:
n_customers = 100000


first_names = ['Max', 'Anna', 'Lukas', 'Sophie', 'Tim', 'Laura', 'Felix', 'Emma', 
               'Paul', 'Mia', 'Leon', 'Hannah', 'Jonas', 'Lena', 'David', 
               'Sarah', 'Ben', 'Julia', 'Noah', 'Lisa']

last_names = ['Müller', 'Schmidt', 'Schneider', 'Fischer', 'Weber', 'Meyer', 
              'Wagner', 'Becker', 'Schulz', 'Hoffmann', 'Koch', 'Bauer',
              'Richter', 'Klein', 'Wolf', 'Schröder', 'Neumann', 'Schwarz',
              'Zimmermann', 'Braun']

cust_names = [f"{random.choice(first_names)} {random.choice(last_names)}" for i in range(n_customers)]

real_addresses = get_random_german_addresses(n_customers=n_customers)

payment_methods = ['credit_card', 'bank_transfer', 'mailed_check', 'electronic_check']

customer_profile = pd.DataFrame({
    'customer_id': [f'CUST_{i:05d}' for i in range(1, n_customers + 1)],
    'customer_name': cust_names,
    'signup_date': pd.date_range(end='2026-05-01', periods=n_customers, freq='12h'),
    'plan_tier': np.random.choice(['Basic_50Mbps', 'Standard_200Mbps', 'Premium_1Gbps'], 
                                  n_customers, p=[0.35, 0.45, 0.2]),
    'address': real_addresses,
    'contract_type': np.random.choice(['monthly', 'annual', '2-year'], 
                                     n_customers, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(payment_methods,
                                     n_customers, p=[0.2, 0.2, 0.25, 0.35]),
    'autopay_enabled': np.random.choice([True, False], n_customers, p=[0.7, 0.3])
})

plan_prices = {'Basic_50Mbps': 49.99, 'Standard_200Mbps': 79.99, 'Premium_1Gbps': 119.99}
customer_profile['monthly_bill'] = customer_profile['plan_tier'].map(plan_prices)
discount_customers = np.random.choice([True, False], n_customers, p=[0.2, 0.8])
customer_profile.loc[discount_customers, 'monthly_bill'] *= 0.9


speed_tiers = {'Basic_50Mbps': 50, 'Standard_200Mbps': 200, 'Premium_1Gbps': 1000}
customer_profile_2 = customer_profile.copy()
customer_profile_2['speed_tier_mbps'] = customer_profile['plan_tier'].map(speed_tiers)
customer_profile['data_usage_gb_last_month'] = np.random.exponential(300, n_customers).round(1)

print("Customer Profile created")

# Chrun Labels generieren
Mit:
* Customer ID (aus den Customer Profile Daten)
* Churned (ob sie abgewandert sind / als abgewandert eingeschätzt sind)
* Churn Gründe (weshalb sie als abgewandert eingeschätzt werden)

In [0]:
churn_rate = 0.15
n_churned = int(n_customers * churn_rate)
churned_customers = np.random.choice(customer_profile['customer_id'], n_churned, replace=False)

churn_labels = pd.DataFrame({
    'customer_id': customer_profile['customer_id'],
    'churned': customer_profile['customer_id'].isin(churned_customers).astype(int)
})

churn_reasons = ['competitor_price', 'poor_service', 'technical_issues', 'relocation', 'price_increase', 'unknown']
churn_labels.loc[churn_labels['churned'] == 1, 'churn_reason'] = np.random.choice(
    churn_reasons, n_churned, p=[0.35, 0.20, 0.20, 0.10, 0.05, 0.1]
)

print("Churn Labels erstellt")

# Connection Logs generieren
Es wird unterschieden zwischen Logs mit technischen Problemen und ohne. Kunden, die in der Churn Label Datei als abgewandert eingeschätzt werden, erhalten mehr Logs mit technischen Problemen als Kunden, die nicht abgewandert sind.
Mit:
* Timestamp (als ID)
* Issue detected (entweder 'none' oder 'error:' mit Spezifikation des Fehlers)
* Customer ID (bei welchem Kunden dieser Fehler aufgetreten ist)
* Speed Measured Mbps (welche Geschwindigkeit zu der Zeit gemessen wurde)
* Packet Loss Percent (prozentuale Anzahl der verlorengegangene Pakete)
* Latency ms (wie hoch die Latency gemessen wurde)
* Downtime Minuten (bei einem Totalausfall, wie lange die Verbindung unterbrochen wurde)
* Connection Drops Count (wie häufig die Verbindung bei einem auftretenden Problem unterbrochen wurde)

In [0]:
connection_logs = []
log_id = 1

# Dictionary um technische Probleme pro Kunde zu tracken
customer_technical_issues = {}

for customer_id in customer_profile['customer_id']:
    customer_data = customer_profile_2[customer_profile['customer_id'] == customer_id].iloc[0]
    is_churned = churn_labels[churn_labels['customer_id'] == customer_id]['churned'].values[0]
    
    # Initialisiere Issue-Liste für diesen Kunden
    customer_technical_issues[customer_id] = []
    
    # Mehr Logs für Kunden mit Problemen
    n_logs = random.randint(20, 50)
    
    for i in range(n_logs):
        timestamp = datetime(2026, 5, 1) - timedelta(
            days=random.randint(1, 90),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
        
        # Baseline Qualität (gut)
        speed_factor = random.uniform(0.85, 1.0)
        packet_loss = random.uniform(0, 2)
        latency = random.uniform(10, 40)
        downtime = 0
        drops = 0
        
        # Zufällige technische Probleme mit bestimmter Wahrscheinlichkeit
        problem_type = None
        
        # 20% Chance für technische Probleme (höher bei abgewanderten Kunden)
        problem_chance = 0.35 if is_churned else 0.15
        
        if random.random() < problem_chance:
            problem_type = np.random.choice([
                'slow_speed',
                'connection_drops',
                'high_latency',
                'outage',
                'packet_loss'
            ], p=[0.3, 0.15, 0.25, 0.1, 0.20])
            
            # Simuliere verschiedene Problemtypen
            if problem_type == 'slow_speed':
                speed_factor = random.uniform(0.3, 0.7)  # Nur 30-70% des Speeds
                latency = random.uniform(50, 150)
                packet_loss = random.uniform(1, 2)
                
            elif problem_type == 'connection_drops':
                drops = random.randint(3, 15)
                packet_loss = random.uniform(2, 8)
                latency = random.uniform(80, 200)
                speed_factor = random.uniform(0.6, 0.9)
                
            elif problem_type == 'high_latency':
                latency = random.uniform(150, 500)
                speed_factor = random.uniform(0.7, 0.95)
                
            elif problem_type == 'outage':
                downtime = random.randint(60, 300)
                speed_factor = 0
                drops = random.randint(50, 200)
                packet_loss = 100
                latency = 0
                
            elif problem_type == 'packet_loss':
                packet_loss = random.uniform(5, 25)
                latency = random.uniform(50, 120)
                speed_factor = random.uniform(0.5, 0.8)
                drops = random.randint(1, 5)
            
            # Speichere dieses Problem-Event
            customer_technical_issues[customer_id].append({
                'timestamp': timestamp,
                'problem_type': problem_type
            })
            

        connection_logs.append({
            'timestamp': timestamp,
            'issue_detected': 'error: ' + problem_type if problem_type else 'none',
            'customer_id': customer_id,
            'speed_measured_mbps': round(customer_data['speed_tier_mbps'] * speed_factor, 1),
            'packet_loss_percent': round(packet_loss, 2),
            'latency_ms': round(latency, 1),
            'downtime_minutes': downtime,
            'connection_drops_count': drops
        })
        
        log_id += 1

connection_quality_logs = pd.DataFrame(connection_logs)

print(f"Connection Logs erstellt: {len(connection_quality_logs)} Logs")
print(f"Davon Logs mit Problemen: {len(connection_quality_logs[connection_quality_logs['issue_detected'] != 'none'])}")

# Agents generieren
Mitarbeiter der Firma, die im direkten Kundenkontakt stehen.

Mit:
* ID (Format: 'AGENT_###')
* Name (Auswahl aus den Namen, die bei den Kunden definiert wurden)
* Employment Date (alle 45 Tage rückführend von dem 1.3.26)
* Experience Level (Einstufung des Erfahrungslevels der Mitarbeiter)
* Monthly Salary Eur (Gehalt basierend auf dem Erfahrungslevel)

In [0]:
n_agents = 20

agent_names = [f"{first_names[i]} {last_names[i]}" for i in range(n_agents)]

agent_profile = pd.DataFrame({
    'agent_id': [f'AGENT_{i:03d}' for i in range(1, n_agents + 1)],
    'agent_name': agent_names,
    'employment_date': pd.date_range(end='2026-03-01', periods=n_agents, freq='45D'), 
    'experience_level': np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], 
                                        n_agents, p=[0.3, 0.45, 0.2, 0.05])
})

# Gehalt basierend auf Experience Level und Department
salary_base = {
    'Junior': 2800,
    'Mid': 3500,
    'Senior': 4500,
    'Lead': 5500
}

agent_profile['monthly_salary_eur'] = agent_profile.apply(
    lambda row: salary_base[row['experience_level']] + 
                np.random.randint(-200, 300),
    axis=1
)

print("Agent Profile erstellt")

# Daten speichern


In [0]:
customer_profile.to_csv('customer_profile.csv', index=False)
churn_labels.to_csv('churn_label.csv', index=False)
connection_quality_logs.to_csv('connection_quality_log.csv', index=False)
agent_profile.to_csv('agent_profile.csv', index=False)

print(f"\nZusammenfassung:")
print(f"   - Customers: {len(customer_profile)}")
print(f"   - Churned Customers: {n_churned} ({churn_rate*100}%)")
print(f"   - Connection Quality Logs: {len(connection_quality_logs)}")
print(f"     - Mit erkannten Problemen: {(connection_quality_logs['issue_detected'] != 'none').sum()}")
print(f"   - Agents: {len(agent_profile)}")